# Bias Analysis

##  1 Import cleaned data and foundatal check

In [3]:
import sys
from pathlib import Path
import importlib

sys.path.append(str(Path("..").resolve()))

import src.data_clean_notebook as preproc
importlib.reload(preproc)

df = preproc.run_data_quality_pipeline("../data/raw_credit_applications.json")

Final dataset shape: (502, 27)


In [4]:
print("shape:", df.shape)
print("columns:", df.columns.tolist())

shape: (502, 27)
columns: ['applicant_gender', 'applicant_zip_code', 'fin_annual_income', 'fin_credit_history_months', 'fin_debt_to_income', 'fin_savings_balance', 'decision_loan_approved', 'spend_shopping', 'spend_rent', 'spend_alcohol', 'spend_txn_count', 'spend_total', 'spend_mean', 'spend_max', 'spend_unique_cats', 'spend_dining', 'spend_healthcare', 'spend_fitness', 'spend_entertainment', 'spend_insurance', 'spend_travel', 'spend_transportation', 'spend_utilities', 'spend_groceries', 'spend_education', 'spend_adult_entertainment', 'spend_gambling']


In [5]:
# Minimal sanity checks
print("\nRaw gender counts:")
print(df["applicant_gender"].value_counts(dropna=False))

print("\nRaw label counts:")
print(df["decision_loan_approved"].value_counts(dropna=False))


Raw gender counts:
applicant_gender
Male      195
Female    193
F          58
M          53
NaN         3
Name: count, dtype: int64

Raw label counts:
decision_loan_approved
True     292
False    210
Name: count, dtype: int64


## Compute approval rates by gender

In order to compute DI rate, we  drop the three Nan, it will cause the bias, but we will consider this in missing value influence.

We first compute approval/selection rates by group and then compute Disparate Impact (DI). DI is computed on rows with non-missing protected attribute and outcome, because DI is undefined otherwise. We also report group sample sizes.

In [13]:
# Missingness audit on raw applicant_gender (before normalization)

raw_gender = df["applicant_gender"]
miss_raw_gender = raw_gender.isna() | (raw_gender.astype(str).str.strip() == "")

print(f"Missing applicant_gender rate: {miss_raw_gender.mean():.4%}")
print("Counts (missing vs non-missing):")
print(miss_raw_gender.value_counts())

# If y exists, compare approval rate
df["y"] = df["decision_loan_approved"].astype(int)
print("\nApproval rate when raw gender is missing:", df.loc[miss_raw_gender, "y"].mean() if miss_raw_gender.any() else "N/A")
print("Approval rate when raw gender is NOT missing:", df.loc[~miss_raw_gender, "y"].mean())

Missing applicant_gender rate: 0.5976%
Counts (missing vs non-missing):
applicant_gender
False    499
True       3
Name: count, dtype: int64

Approval rate when raw gender is missing: 0.6666666666666666
Approval rate when raw gender is NOT missing: 0.5811623246492986


In [14]:
# Create gender_norm WITHOUT hiding missing values

g = df["applicant_gender"].astype("string").str.strip().str.lower()

gender_map = {
    "m": "male", "male": "male", "man": "male",
    "f": "female", "female": "female", "woman": "female",
}

df["gender_norm"] = g.map(gender_map)

# IMPORTANT: keep missing as <NA> (do NOT fillna here)
# df["gender_norm"] remains NA if raw gender was missing or unrecognized

# (optional) show what values are not mapped
print("gender_norm value counts (including NA):")
print(df["gender_norm"].value_counts(dropna=False))

gender_norm value counts (including NA):
gender_norm
female    251
male      248
NaN         3
Name: count, dtype: int64


In [17]:
# Prepare DI dataset (DI is undefined when gender is missing)

tmp = df[["gender_norm", "y"]].dropna().copy()
tmp["y"] = tmp["y"].astype(int)

print("DI dataset size:", tmp.shape[0])
print(tmp["gender_norm"].value_counts())

group_sizes = tmp["gender_norm"].value_counts()
approval_rates = tmp.groupby("gender_norm")["y"].mean().sort_values(ascending=False)

print("Group sizes:\n", group_sizes)
print("\nApproval rates:\n", approval_rates)

DI dataset size: 499
gender_norm
female    251
male      248
Name: count, dtype: int64
Group sizes:
 gender_norm
female    251
male      248
Name: count, dtype: int64

Approval rates:
 gender_norm
male      0.657258
female    0.505976
Name: y, dtype: float64


In [18]:
priv_group = "male"
unpriv_group = "female"

if priv_group in approval_rates.index and unpriv_group in approval_rates.index:
    di = float(approval_rates.loc[unpriv_group] / approval_rates.loc[priv_group])
    print(f"DI (female/male) = {di:.4f}")
    print(f"80% rule flag (DI < 0.8): {di < 0.8}")
else:
    print("Cannot compute DI: missing male or female group in the DI dataset.")
    print("Available groups:", approval_rates.index.tolist())

DI (female/male) = 0.7698
80% rule flag (DI < 0.8): True


Conclusion — Gender DI (Disparate Impact)

Missingness check: applicant_gender is missing for 3/502 (0.60%) records. The approval rate for the missing-gender rows is 0.667 (2/3) vs 0.581 for non-missing rows. Because the missing group is extremely small, excluding these rows is unlikely to materially change the DI estimate, but we report it transparently.

Group selection rates: On the DI computation subset (n=499 with non-missing gender), the approval/selection rate is 0.657 for male (n=248) and 0.506 for female (n=251).

Disparate Impact (DI): Using the common convention DI = female_rate / male_rate, we obtain DI = 0.7698. This falls below the 0.80 four-fifths threshold, so the 80% rule is triggered, indicating potential adverse impact against the female group in the observed decision outcomes.

Note on interpretation: DI is a screening metric, not a causal proof of discrimination. Next steps include investigating possible proxy drivers (e.g., zip code or financial variables correlated with gender) and checking whether the disparity persists after controlling for relevant risk factors.